# Experimento — STab

STab (Hajiramezanali et al., 2022): dois codificadores MLP com pesos compartilhados e DropConnect, stop-gradient e perda de cosseno; cabeça de regressão logística sobre o embedding congelado. Reimplementação própria. Nesta rodada em CPU a máscara é sorteada por grupo de 16 amostras e a largura vai até 128.

Protocolo comum da pré-proposta: mesmas 5 dobras e mesmo teste de `if702-preprocessamento.ipynb` (via `comum.py`), KS como métrica e critério da busca, 8 hiperparâmetros e o mesmo N de trials no Optuna (`TPESampler(seed=42)`, `MedianPruner`, trial 0 = padrão, estudo em SQLite). Entropia cruzada e MSE entram como um dos 8 hiperparâmetros (`perda`).

In [1]:
import json, sys, numpy as np, pandas as pd
from pathlib import Path
# comum.py fica na raiz do repositorio; funciona rodando da raiz ou da pasta do notebook
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'comum.py').exists())
sys.path[:0] = [str(RAIZ), str(RAIZ / 'experimentos' / 'stab')]
import optuna, comum, modelo as M
folds, final, colunas = comum.carregar_dobras()   # CSV: variável de ambiente CHURN_CSV ou dados/telco_ibm.csv

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Estudo Optuna

Para refazer a busca (leva de dezenas de minutos a horas; é retomável): `st = M.estudo(N)`. Aqui só lemos o estudo salvo.

In [2]:
MODELO = 'stab'
st = optuna.load_study(study_name=MODELO, storage=f'sqlite:///{comum.RESULTADOS}/{MODELO}.db')
df = st.trials_dataframe()[['number','state','value']+[c for c in st.trials_dataframe().columns if c.startswith('params_')]]
df.sort_values('value', ascending=False).head(10)

,number,state,value,params_ativ,params_larg,params_lr,params_otim,params_p,params_perda,params_prof,params_wd
30,30,PRUNED,0.566637,relu,128,0.000678,adam,0.949579,ce,2,0.000114
15,15,PRUNED,0.561542,relu,128,0.000324,adam,0.929989,ce,4,0.000001
29,29,PRUNED,0.560198,relu,128,0.005462,adam,0.940755,ce,3,0.000002
24,24,PRUNED,0.558111,relu,128,0.001496,adam,0.907202,ce,3,0.000012
21,21,PRUNED,0.556882,relu,128,0.000363,sgd,0.664442,ce,4,0.000023
25,25,PRUNED,0.553933,relu,128,0.000902,adam,0.768456,ce,3,0.000005
16,16,PRUNED,0.547490,relu,128,0.005754,sgd,0.643248,ce,3,0.000001
13,13,PRUNED,0.546970,relu,128,0.000138,adam,0.551108,ce,3,0.000005
0,0,COMPLETE,0.545273,relu,128,0.001000,adam,0.900000,ce,4,0.000010
28,28,COMPLETE,0.545231,relu,128,0.002682,adam,0.835145,ce,4,0.000003


## 2. Resultados finais (`resultados/<modelo>.json`)

In [3]:
r = json.load(open(f'{comum.RESULTADOS}/{MODELO}.json'))
print('Melhor configuração:', r['melhor_config'])
e = r['estudo']
print(f"KS trial 0 = {e['ks_trial0']:.4f} -> melhor trial = {e['melhor_ks']:.4f} | completos {e['n_completos']}, podados {e['n_podados']}")
print('Importâncias (Optuna):', {k: round(v,3) for k,v in e['importancias'].items()})
pd.DataFrame(r['ce_vs_mse']['teste']).T[['ks','auroc','ce','mse','ce_corr','f1','precision','recall']]

Melhor configuração: {'prof': 4, 'larg': 128, 'ativ': 'relu', 'otim': 'adam', 'lr': 0.001, 'p': 0.9, 'wd': 1e-05, 'perda': 'ce'}
KS trial 0 = 0.5453 -> melhor trial = 0.5453 | completos 21, podados 9
Importâncias (Optuna): {'prof': 0.338, 'lr': 0.261, 'wd': 0.131, 'otim': 0.116, 'p': 0.064, 'ativ': 0.059, 'larg': 0.03, 'perda': 0.001}


,ks,auroc,ce,mse,ce_corr,f1,precision,recall
ce,0.543773,0.848133,0.485114,0.162193,0.413815,0.634436,0.525281,0.800857
mse,0.546139,0.847636,0.485924,0.162344,0.414344,0.637857,0.534783,0.79015


## 3. Entropia cruzada × MSE (mesma configuração, mesmas dobras e sementes)

In [4]:
c = r['ce_vs_mse']
print(f"ΔKS validação (CE−MSE) = {c['delta_ks_val_media']:+.4f} ± {c['delta_ks_val_dp']:.4f} | CE melhor em {c['dobras_ce_melhor']}/5 dobras | Wilcoxon p = {c['wilcoxon_p']:.3f}")
print(f"ΔKS teste (CE−MSE) = {c['delta_ks_teste']:+.4f}  IC95% {np.round(c['delta_ks_teste_ic95'],4)}")
pd.DataFrame({k: {m: v[m][0] for m in v} for k, v in c['resumo'].items()})

ΔKS validação (CE−MSE) = +0.0040 ± 0.0069 | CE melhor em 5/5 dobras | Wilcoxon p = 0.062
ΔKS teste (CE−MSE) = -0.0024  IC95% [-0.0085  0.0057]


,ce,mse
ks,0.531567,0.527589
auroc,0.837139,0.837156
ce_raw,0.498175,0.498779
mse_raw,0.166732,0.166799
ce_corr,0.425827,0.425827
mse_corr,0.138101,0.138082
tr_ce,0.480124,0.483186
tr_mse,0.157514,0.157916


## 4. Limitações desta rodada

- Rodada em CPU, N = 30 trials; versão final pretendida: GPU T4 (`colab_stab_tabkan.ipynb`), N comum do grupo.
- Early stopping usa a mesma dobra que escolhe a configuração (KS de validação levemente otimista).
- O KS do teste é de um comitê das 5 dobras × 5 sementes, avaliado uma única vez.